In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/referencias/"
normalize_table = str.maketrans({"ã": "a", "á": "a", "à": "a", "â": "a", "é": "e", "ê": "e", "í": "i", "ì": "i", "î": "i", "ó": "o", "ò": "o", "ô": "o", "õ": "o", "ú": "u", "ù": "u", "û": "u", "ç": "c", " ": "_", "(": "", ")": "", '"': ""})
TABELA = "voebem.bronze."

In [0]:
aerodromos = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", "")
    .option("escape", "")
    .option("encoding", "ISO-8859-1")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO + "AerodromosPublicos.csv")
)


for cols in aerodromos.columns:
    novo = cols.translate(normalize_table).lower()
    # print(f"\n{cols!r} -> {novo!r}")
    aerodromos = aerodromos.withColumnRenamed(cols, novo)

In [0]:
aerodromos = aerodromos.withColumn(
    "_ingerido_em", F.current_timestamp()
)

aerodromos.write.format("delta").mode("overwrite").options(overwriteSchema="true").saveAsTable(TABELA + "aerodromos")

In [0]:
empresas_nacionais = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", "")
    .option("escape", "")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO + "pda_empresas_aereas_nacionais.csv")
)

for cols in empresas_nacionais.columns:
    novo = cols.translate(normalize_table).lower()
    empresas_nacionais = empresas_nacionais.withColumnRenamed(cols, novo)

empresas_nacionais = empresas_nacionais.withColumn(
    "_ingerido_em", F.current_timestamp()
)
empresas_nacionais.write.format("delta").mode("overwrite").options(overwriteSchema="true").saveAsTable(
    TABELA + "empresas_nacionais"
)

In [0]:
empresas_estrangeiras = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", "")
    .option("escape", "")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO + "pda_empresas_aereas_estrangeiros.csv")
)

for cols in empresas_estrangeiras.columns:
    novo = cols.translate(normalize_table).lower()
    empresas_estrangeiras = empresas_estrangeiras.withColumnRenamed(cols, novo)

empresas_estrangeiras = empresas_estrangeiras.withColumn(
    "_ingerido_em", F.current_timestamp()
)
empresas_estrangeiras.write.format("delta").mode("overwrite").options(overwriteSchema="true").saveAsTable(
    TABELA + "empresas_estrangeiras"
)

In [0]:
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codigos_df = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")

codigos_df.write.format("delta").mode("overwrite").options(overwriteSchema="true").saveAsTable(
    TABELA + "codigos_operacao"
)